# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane 2 (Refresh / Content Opportunity Scoring) is a **ranking / scoring** task at its top level:
the deliverable is a ranked review queue, and the metric that matters is "how many of the top-K
recommendations were worth an editor's time" (precision@K), not "did we classify every row
correctly."

Underneath that ranking sits a **classification** sub-task, because the starter pipeline (and my
own approach) scores each page with `P(page is a decline/opportunity candidate)` and uses that
probability to build the ranking - the same pattern the starter pipeline uses
(`scripts/03_train_model.py` trains a classifier, `04_evaluate_and_export.py` turns its
probabilities plus a rule-based baseline into a ranked `final_refresh_score`). So: classification
underneath, ranking/scoring on top - not clustering, since I'm not grouping unlabeled pages into
types, and not pure unsupervised signal analysis, since the output is an ordered action list.

## 2. Target or proxy

**Proxy target for now:** `is_declining_label = (trend_direction == "down")`, exactly as the
starter pipeline defines it. This is a **defined-rule proxy, not an observed future outcome** -
`trend_direction` (and the `trend_pct` it's computed from) describes the *current* 90-day window,
not what happens *next*. Per `skills/flyrank/flyrank-data/SKILL.md`, `trend_direction` and
`trend_pct` are the label's own ingredients, so they can never also be features - that would be
circular.

**Why I'm keeping it as the Week 2 proxy anyway:** it's the only label the starter CSV supports
(one row per page, one snapshot in time - no forward-looking window to measure "declined further"
against). The lane guide is explicit that a stronger capstone target looks like
`features from prior 90 days -> decline or recovery over next 30 days`, built from the warehouse's
daily fact table (`fact_content_daily_performance`) once I define a feature window and a separate,
later target window with no overlap. I'm flagging that upgrade as a Week 3-4 data-contract task,
not doing it yet - Week 2's job is to frame the task honestly with the data I have today.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print()
print(f"declining-label rate: {df['is_declining_label'].mean():.3f}")
print()
print("trend_direction breakdown behind the label:")
print(df["trend_direction"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

declining-label rate: 0.542

trend_direction breakdown behind the label:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

**Primary metric: precision@50.** The decision this supports is "which ~50 pages does an editor
open this week" (a fixed, small review capacity), so the number that matters is: of the top 50
pages the ranking puts first, how many are genuinely worth reviewing? That is exactly what
precision@K measures, and it's the metric the lane guide names for ranking/scoring tasks.

I'm choosing this over plain accuracy or ROC AUC because both of those score the *whole* ranked
list evenly, while an editor only ever reaches the top of it — a model that is excellent on
average but weak at the very top is worse for this decision than one that is merely good at the
top and mediocre lower down.

Reported alongside precision@50 (not fixed yet, but named as secondary): recall, so I can see how
much true opportunity a fixed capacity of 50 leaves on the table, and average precision, to keep
comparisons stable if the review capacity changes later.

**A number I can already defend today**, straight from this repo's own committed
`outputs/model_report.md` (generated by the starter pipeline on this same CSV, with
client-holdout validation - whole clients withheld from training):

In [2]:
import json

# outputs/model_report.md ships in the repo already, generated by scripts/run_all.py.
# Pulling the model-comparison table straight from it rather than retyping numbers by hand.
with open("../../outputs/model_report.md") as f:
    report = f.read()

start = report.index("## Model Comparison")
end = report.index("## Final Queue")
print(report[start:end].strip())

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 4. The unit of analysis, as a real dataframe

One row = one content item (`content_id`), described by its own trailing-90-day search and
engagement history. That's the grain the ranking is built on: each row is a candidate the model
scores and the queue ranks, one page at a time.

Below is my lane's slice: the columns I'd actually feed a model or scorer (numeric/categorical
observed and derived signals) — with `trend_direction`, `trend_pct`, `content_id`, and `client_id`
kept visible for context but marked out as non-features (label ingredients and join keys,
per the data skill's gotchas).

In [3]:
feature_cols = [
    "search_volume", "competition", "competition_level", "cpc",
    "content_type", "main_intent", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "days_since_last_update", "freshness_tier",
    "word_count_tier", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "impression_tier", "position_tier",
]
context_cols = ["content_id", "client_id"]          # join keys only, never features
label_ingredient_cols = ["trend_direction", "trend_pct"]  # the label is built FROM these

lane_slice = df[context_cols + feature_cols + label_ingredient_cols + ["is_declining_label"]]

print(f"one row = one content item; shape: {lane_slice.shape}")
lane_slice.head(3)

one row = one content item; shape: (30000, 31)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


## 5. Why ML beats a fixed rule here

Three things push this past what a single if-statement can do:

1. **A hand-written rule barely fires.** From Week 1's numbers: the obvious staleness rule
   (`days_since_last_update >= 180` and `impressions_90d >= 500`) flags only 17 of 30,000 rows
   (0.1%) — nowhere near enough to fill a weekly review queue, and it ignores every page that's
   declining for a reason other than staleness.
2. **Meanwhile the raw signals are everywhere, but noisy.** 54.2% of pages are currently trending
   down and 32.5% look like low-CTR opportunities by themselves — far too many for one rule to
   safely flag as "review this," so the real task is *weighing and combining* many weak, partly
   overlapping signals (trend, position, CTR, freshness, volume, content depth) into one ranking,
   which is a pattern-combination problem, not a single-threshold one.
3. **The starter pipeline already shows the gap, on this exact data.** The rule-based baseline
   gets precision@50 = 0.240 (about 12 of the top 50 right); a random forest trained on the same
   rows gets 0.740 (about 37 of 50) — evidence, on this slice, that learning how to combine the
   signals ranks meaningfully better than a fixed formula, though this is one holdout split on a
   30k-row anonymized sample, not a claim that generalizes yet to the full warehouse.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.